In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, mean_absolute_error, mean_squared_error
from sklearn.preprocessing import LabelEncoder
import numpy as np

In [7]:
# Change working directory to project root
import os
os.chdir('..')

#read the cleaned csv file
gym_df = pd.read_csv('data/processed/rutgers_gym_cleaned.csv')

#look at data
gym_df.head()
print(gym_df.columns.tolist())

['gym_name', 'day_of_week', 'hour', 'busyness_percent', 'hour_24', 'is_weekend', 'semester_week', 'is_exam_week', 'weather', 'capacity_label']


In [10]:
#initialize the label encoder
label_encoder = LabelEncoder()

#convert gym names into numbers 
gym_df['gym_name_encoded'] = label_encoder.fit_transform(gym_df['gym_name'])

#convert day names into numbers 
gym_df['day_of_week_encoded'] = label_encoder.fit_transform(gym_df['day_of_week'])

#convert weathers into numbers 
gym_df['weather_encoded'] = label_encoder.fit_transform(gym_df['weather'])

#convert capacity into numbers -> target 
gym_df['capacity_label_encoded'] = label_encoder.fit_transform(gym_df['capacity_label'])

#check if encoding worked
print(gym_df[['gym_name', 'gym_name_encoded', 'day_of_week', 'day_of_week_encoded', 'capacity_label', 'capacity_label_encoded']].head())


             gym_name  gym_name_encoded day_of_week  day_of_week_encoded  \
0  College Avenue Gym                 0      Monday                    1   
1  College Avenue Gym                 0      Monday                    1   
2  College Avenue Gym                 0      Monday                    1   
3  College Avenue Gym                 0      Monday                    1   
4  College Avenue Gym                 0      Monday                    1   

  capacity_label  capacity_label_encoded  
0            Low                       1  
1            Low                       1  
2            Low                       1  
3            Low                       1  
4            Low                       1  


In [12]:
#select columns as features for model
features = [
    'gym_name_encoded',
    'day_of_week_encoded',
    'hour_24',
    'is_weekend',
    'semester_week',
    'is_exam_week',
    'weather_encoded'
]

#define X for input
X = gym_df[features]

#define y for target
y = gym_df['capacity_label_encoded']

print('Input shape:', X.shape)
print('Target shape:', y.shape)

Input shape: (331, 7)
Target shape: (331,)


In [13]:
#split data training:testing -> 80%:20%
#random_state=42 ensures consistent result every run
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print('Training rows:', len(X_train))
print('Testing rows:', len(X_test))

Training rows: 264
Testing rows: 67


In [15]:
#initialize Random Forest
#n_estimators=100 -> 100 decision trees and votes on result
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)

#train model on test data
rf_model.fit(X_train, y_train)

print('Model trained successfully')


Model trained successfully


In [16]:
#use traind model to predict test data
y_pred = rf_model.predict(X_test)

#calculate accuracy
accuracy = accuracy_score(y_test, y_pred)

#calculate MAE (average diff between pred. and actual)
mae = mean_absolute_error(y_test, y_pred)

#calculate RMSE 
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print('Accuracy:', round(accuracy * 100, 2), '%')
print('MAE:', round(mae, 3))
print('RMSE:', round(rmse, 3))

Accuracy: 76.12 %
MAE: 0.358
RMSE: 0.773


In [ ]:
#feature importance scores from model
importances = rf_model.feature_importances_

#plot bar chart
plt.figure(figsize=(10, 6))
plt.bar(features, importances)

#title and labels
plt.title('Feature Importance- Random Forest')
plt.xlabel('Feature')
plt.ylabel('Importance Score')

#remove overlapping with x-axis 
plt.xticks(rotation=45, ha='right')

#space adjustment
plt.tight_layout()

#display plot
plt.show()